In [ ]:
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

from gaitutils.stats import collect_trial_data
from gaitutils.trial import Trial

In [ ]:
NORMAL_DATA_FNAME = '/home/andrey/scratch/multilevel-surgery/TD_normaldata_all.xlsx'
CATALOG_FNAME = '/home/andrey/scratch/multilevel-surgery/files.xlsx'
DATA_DIR = '/home/andrey/scratch/multilevel-surgery'

NORM_TRIAL_LEN = 101    # number of points to which each cycle is resampled

VAR_NAME_MAP = {
    'AnkleAnglesX': 'AnkleAngles (1)',
    'KneeAnglesX': 'KneeAngles (1)',
    'HipAnglesX': 'HipAngles (1)',
    'HipAnglesY': 'HipAngles (2)',
    'HipAnglesZ': 'HipAngles (3)',
    'PelvisAnglesX': 'PelvisAngles (1)',
    'PelvisAnglesY': 'PelvisAngles (2)',
    'PelvisAnglesZ': 'PelvisAngles (3)',
    'FootProgressAnglesZ': 'FootProgressAngles (3)'
}

## Read and prepare the normal data

In [ ]:
df_normal = pd.read_excel(NORMAL_DATA_FNAME, header=0, skiprows=[1, 2])
norm_data = {}

for var_name in VAR_NAME_MAP:
    vals_low = df_normal[VAR_NAME_MAP[var_name]].to_numpy()
    vals_high = df_normal[VAR_NAME_MAP[var_name] + '.1'].to_numpy()

    x_old = np.linspace(0, 1, len(vals_low))
    x_new = np.linspace(0, 1, NORM_TRIAL_LEN)

    norm_data[var_name] = np.interp(x_new, x_old, (vals_low + vals_high) / 2)


In [ ]:
all_gps = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))  # patient_id -> condition -> R/L -> list of values

In [ ]:
df = pd.read_excel(CATALOG_FNAME, header=0)

for idx, row in df.iterrows():
    fullpath = Path(DATA_DIR) / row['subject'] /row['relpath']
    assert fullpath.is_dir(), f"Directory {fullpath} does not exist"

    tags = row['tagfilter'].split(',')
    tags = [tag.strip() for tag in tags]
    print(f"Processing {fullpath} with tags {tags} ...")

    for c3d_file in fullpath.glob('*.c3d'):
        trial = Trial(c3d_file)

        tag_hits = ((tag in trial.eclipse_data['DESCRIPTION']) or (tag in trial.eclipse_data['NOTES']) for tag in tags)

        #print(f"Checking tags for file {c3d_file} ...")
        #print(f"Tags to check: {tags}")
        #print(f"Tag hits: {list(tag_hits)}")

        if any(tag_hits):
            print(f'File {c3d_file} is tagged, collecting data...')
            data, cycles = collect_trial_data(trial, analog_envelope=True, force_collect_all_cycles=True, fp_cycles_only=False)

            for side in ['R', 'L']:

                var_mean_sqs = []
                for var_name in VAR_NAME_MAP:
                    var_mean_sq = ((data['model'][f'{side}{var_name}']) ** 2 - (norm_data[var_name] ** 2)).mean(axis=1)
                    var_mean_sqs.append(var_mean_sq)

                gps_sq = np.mean(np.stack(var_mean_sqs), axis=0)
                gps = np.sqrt(gps_sq)

                all_gps[row['subject']][row['condition']][side].extend(gps.tolist())
